## 对话管理
- 直接追加消息


In [3]:
import os
from typing import override

from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
from langchain_openai import ChatOpenAI

load_dotenv(override= True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = ChatDeepSeek(
    #api_key= DEEPSEEK_API_KEY,
    #base_url= DEEPSEEK_BASE_URL,
    model= "deepseek-v4-flash"
)

conversation = []
# 第一次
conversation.append({"role": "user", "content": "我叫张三"})
response1 = model.invoke(conversation)
# 关键：保存 AI 回复
conversation.append({"role": "assistant", "content": response1.content})
# 第二次（传递完整历史）
conversation.append({"role": "user", "content": "我叫什么？"})
response2 = model.invoke(conversation)

### 长对话保留最近几轮消息


In [1]:
def keep_recent_messages(messages, max_pairs=2):
    """
    保留最近 max_pairs 轮对话
    :param messages: 对话消息列表
    :param max_pairs: 最大轮数
    :return: 优化后的对话消息列表
    """
    # 1. 保留系统消息
    system_msg = [msg for msg in messages if msg["role"] == "system"]

    # 2. 从后往前找最近 max_pairs 轮 user 消息的起始位置
    user_count = 0
    start_index = 0

    for i in range(len(messages) - 1, -1, -1):
        if messages[i]["role"] == "user":
            user_count += 1
            if user_count == max_pairs:
                start_index = i
                break

    # 3. 拼接：system_msg + 从 start_index 开始的所有消息
    return system_msg + messages[start_index:]

In [4]:
# 初始化
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]
# 第 1 轮
long_conversation.append({"role": "user", "content": "什么是列表？用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})
# 第 2 轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别？用一句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})
# 第 3 轮
long_conversation.append({"role": "user", "content": "什么是字典呢？用一句解释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})
print(f"原始消息数: {len(long_conversation)}")
# 优化：只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")
# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么？"})
# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 你问的第一个问题是：“列表和元组有什么区别？用一句解释”
